# 5 — Final 480-row validation, analysis, and emergency export
Do not interpret partial results. This notebook refuses to analyze unless all 456 new episode artifacts and all 480 result rows are present and valid.


In [ ]:
import csv, subprocess, os, hashlib
from pathlib import Path
R=Path.home()/"async-vla-latency-bench"; OUT=Path.home()/"stage1"; PY=Path.home()/"venv-stage1-ood/bin/python"
manifest=list(csv.DictReader(open(OUT/"stage1_manifest.csv"))); planned_new={r['run_id'] for r in manifest if r['reuse_stage0'].lower()!='true'}; done={p.stem for p in (OUT/"episodes").glob('*.json')}
assert len(manifest)==480 and len(planned_new)==456 and len(done & planned_new)==456, (len(manifest),len(planned_new),len(done & planned_new))
subprocess.run([str(PY),"-m","async_vla_benchmark.scripts.validate_stage1","--manifest",str(OUT/"stage1_manifest.csv"),"--output-dir",str(OUT)],cwd=R,check=True)
subprocess.run([str(PY),"-m","async_vla_benchmark.scripts.analyze_stage1","--results",str(OUT/"stage1_episode_results.csv"),"--output-dir",str(OUT/"analysis")],cwd=R,check=True)
subprocess.run([str(PY),"-m","async_vla_benchmark.scripts.plot_stage1","--results",str(OUT/"stage1_episode_results.csv"),"--tables-dir",str(OUT/"analysis"),"--output-dir",str(OUT/"analysis")],cwd=R,check=True)
print("PASS: complete Stage 1 analysis generated")


In [ ]:
# Capture environments and create a portable archive. Download it immediately.
subprocess.run([str(Path.home()/"venv-stage1-id/bin/pip"),"freeze"],stdout=open(OUT/"pip_freeze_id.txt","w"),check=True)
subprocess.run([str(Path.home()/"venv-stage1-ood/bin/pip"),"freeze"],stdout=open(OUT/"pip_freeze_ood.txt","w"),check=True)
archive=Path.home()/"stage1_results.tar.gz"
subprocess.run(["tar","-czf",str(archive),"-C",str(Path.home()),"stage1"],check=True)
h=hashlib.sha256(archive.read_bytes()).hexdigest(); print(archive,archive.stat().st_size,"bytes"); print("sha256",h)
print("DOWNLOAD THIS ARCHIVE NOW, then paste validator output, artifact listing, and checksum here.")
